### Imports

In [18]:
import sys
import os

sys.path.append(os.path.abspath("..")) 

import json
import pandas as pd
from datetime import datetime
import re
from data_class.raw_data import RawData
from tqdm.auto import tqdm

c:\Users\jomka\OneDrive\Desktop\THESIS\data_gathering_repo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Helper functions for cleaning

In [22]:
def clean_date(date: str) -> str:
    """Convert various date formats to ISO format"""
    
    # Remove "FIRST PUBLISHED" prefix
    date_str = re.sub(r'FIRST PUBLISHED\s+', '', date, flags=re.IGNORECASE)

    # Remove "UPDATED" prefix
    date_str = re.sub(r'UPDATED\s+', '', date_str, flags=re.IGNORECASE)
    
    try:
        # Parse date (format: "4 NOVEMBER 2022")
        date_obj = datetime.strptime(date_str, '%d %B %Y')
        return date_obj.isoformat()
    except ValueError as e:
        print(f"Could not parse date: {date_str}")
        raise e

def clean_text(text: str | None) -> str | None:
    """Normalize whitespace and line breaks in text"""
    if not text:
        return text

    # Replace multiple spaces with single space
    text = re.sub(r'\s+', ' ', text)
    # Normalize line breaks
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    # Remove leading/trailing whitespace
    text = text.strip()
    
    return text

def clean_article(article: RawData):
    """Clean a single article entry"""
    cleaned: RawData = article.copy()
    
    # Clean date
    cleaned['publish_date'] = clean_date(cleaned['publish_date'])
    
    # Clean text fields
    for field in ['title', 'content', 'claim', 'verdict']:
        if field in cleaned and cleaned[field]:
            cleaned[field] = clean_text(cleaned[field])
    
    # Ensure authors is a list
    if 'authors' in cleaned and cleaned['authors'] is None:
        cleaned['authors'] = []

    # Add source bias from: https://mediabiasfactcheck.com/full-fact-uk/
    cleaned["source_bias"] = "LEAST-BIASED"
    
    return cleaned

### Load in dataset

In [23]:
with open('../outputs/fullfact-factcheck.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

### Clean 

In [24]:
# Clean all articles
cleaned_data = [clean_article(article) for article in tqdm(data, desc="Cleaning articles")]

# Split into fact-check and fact-check-no-verdict
fact_checks_with_verdict = [
    article for article in cleaned_data 
    if article.get('type') == 'fact-check'
]

fact_checks_without_verdict = [
    article for article in cleaned_data 
    if article.get('type') == 'fact-check-no-verdict'
]













Cleaning articles: 100%|██████████| 8117/8117 [00:01<00:00, 6326.47it/s]


### Convert to DF and inspect

In [26]:
df_with_verdict = pd.DataFrame(fact_checks_with_verdict)
df_without_verdict = pd.DataFrame(fact_checks_without_verdict)

print(f"Articles with verdict: {len(fact_checks_with_verdict)}")
print(f"Articles without verdict: {len(fact_checks_without_verdict)}")
print(f"\nSample of cleaned data:")
print(df_with_verdict[['title', 'content', 'publish_date', 'source_bias']].head())

Articles with verdict: 7002
Articles without verdict: 1115

Sample of cleaned data:
                                               title  \
0  The government won’t be taking £90 from people...   
1  John Lewis isn’t giving away expensive cast ir...   
2  No, the government isn’t introducing a £500 ‘C...   
3  Video compilation includes faked clips of Isra...   
4         Are one in six Scots waiting for NHS care?   

                                             content         publish_date  \
0  Videos viewed hundreds of thousands of times o...  2025-12-09T00:00:00   
1  John Lewis is not giving away Staub cast iron ...  2025-12-09T00:00:00   
2  Videos shared hundreds of times on social medi...  2025-12-08T00:00:00   
3  A viral video compilation that supposedly show...  2025-12-05T00:00:00   
4  Scottish Labour, its leader Anas Sarwar and Pr...  2025-12-05T00:00:00   

    source_bias  
0  LEAST-BIASED  
1  LEAST-BIASED  
2  LEAST-BIASED  
3  LEAST-BIASED  
4  LEAST-BIASED  


### Save cleaned data to separate JSON files

In [27]:
output_dir = '../outputs_clean/fullfact'

with open(f'{output_dir}/fullfact_with_verdict.json', 'w', encoding='utf-8') as f:
    json.dump(fact_checks_with_verdict, f, indent=2, ensure_ascii=False)

with open(f'{output_dir}/fullfact_without_verdict.json', 'w', encoding='utf-8') as f:
    json.dump(fact_checks_without_verdict, f, indent=2, ensure_ascii=False)

print("Cleaning complete! Files saved.")

Cleaning complete! Files saved.
